In [26]:
# PyTorch 9

# Basic CNN

In [27]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device

device(type='cuda')

In [28]:
import pandas as pd
import torch.nn as nn

from torch.utils.data import Dataset,DataLoader
import torchmetrics

In [29]:
path = "/content/drive/MyDrive/fmnist_full_dataset/fashion-mnist_train.csv"

df = pd.read_csv(path)
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
X = df.iloc[: ,1:].values
y = df.iloc[:,0].values

In [31]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=38,stratify=y)

In [32]:
X_train = X_train/255.0
X_test = X_test/255.0

In [41]:
class FashionDataset(Dataset):

  def __init__(self,x,y):
    self.x = torch.tensor(x).float().reshape(-1,1,28,28)
    self.y = torch.tensor(y).long()

  def __len__(self):
    return len(self.x)

  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]

In [42]:
train_dataset = FashionDataset(X_train,y_train)
test_dataset = FashionDataset(X_test,y_test)

In [43]:
train_dataloader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_dataloader = DataLoader (test_dataset,batch_size=32,pin_memory=True)

In [44]:
class BasicCNN(nn.Module):

  def __init__(self,channel):
    super().__init__()

    self.feat_extractor = nn.Sequential(
        nn.Conv2d(in_channels=channel,
                  out_channels=32,
                  kernel_size=3,
                  padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(32,64,3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p=0.3),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.3),

        nn.Linear(64,10)
    )

  def forward(self,x):
    x = self.feat_extractor(x)
    x = self.classifier(x)
    return x


In [45]:
import torch.optim as optim

In [48]:
model = BasicCNN(1)
model.to(device)

lr = 0.001
epochs = 50

optimizer = optim.Adam(model.parameters(),lr=lr)

criterion = nn.CrossEntropyLoss()

In [49]:
for epoch in range(epochs):

  model.train()

  for batch_x , batch_y in train_dataloader:

    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output,batch_y)

    loss.backward()

    optimizer.step()

  print(f"Epoch: {epoch+1} , Loss: {loss.item()}")

Epoch: 1 , Loss: 0.7771207690238953
Epoch: 2 , Loss: 0.6667912602424622
Epoch: 3 , Loss: 0.19115261733531952
Epoch: 4 , Loss: 0.28073275089263916
Epoch: 5 , Loss: 0.38532495498657227
Epoch: 6 , Loss: 0.4170284569263458
Epoch: 7 , Loss: 0.11203042417764664
Epoch: 8 , Loss: 0.07850717753171921
Epoch: 9 , Loss: 0.15057697892189026
Epoch: 10 , Loss: 0.1713830828666687
Epoch: 11 , Loss: 0.2869100272655487
Epoch: 12 , Loss: 0.11976947635412216
Epoch: 13 , Loss: 0.11323269456624985
Epoch: 14 , Loss: 0.15524253249168396
Epoch: 15 , Loss: 0.02106228470802307
Epoch: 16 , Loss: 0.0811300277709961
Epoch: 17 , Loss: 0.1048734188079834
Epoch: 18 , Loss: 0.10971972346305847
Epoch: 19 , Loss: 0.24945886433124542
Epoch: 20 , Loss: 0.01663939282298088
Epoch: 21 , Loss: 0.1671592891216278
Epoch: 22 , Loss: 0.21500110626220703
Epoch: 23 , Loss: 0.10669160634279251
Epoch: 24 , Loss: 0.29071420431137085
Epoch: 25 , Loss: 0.11830990016460419
Epoch: 26 , Loss: 0.010065489448606968
Epoch: 27 , Loss: 0.13689360

In [50]:
model.eval()

BasicCNN(
  (feat_extractor): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [51]:
from torchmetrics import Accuracy

metric = Accuracy(task="multiclass",num_classes=10).to(device)

In [53]:
# test evaluation

with torch.no_grad():

    test_loss = 0

    for batch_x, batch_y in test_dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        output = model(batch_x)


        loss = criterion(output, batch_y)

        test_loss += loss.item()

        metric(output, batch_y)


    avg_test_loss = test_loss / len(test_dataloader)

    final_metric = metric.compute()

    print(f"Test Loss: {avg_test_loss:.4f}, Accuracy: {final_metric:.4f}")

    metric.reset()

Test Loss: 0.4215, Accuracy: 0.9251
